# 🚀 Startup Success Prediction
### End-to-end ML pipeline: Data Import → EDA → Feature Engineering → Modeling → Evaluation

---

## 🗺️ What This Notebook Does

This notebook builds a **machine learning model that predicts whether a startup will succeed** (get acquired or go public via IPO) based on real Crunchbase investment data.

| Step | What happens |
|------|--------------|
| 1 | Install and import all required libraries |
| 2 | Download the dataset from Kaggle using `kagglehub` |
| 3 | Load the CSV and inspect its structure |
| 4 | Clean the data and create the target variable (success/failure) |
| 5 | Visualize patterns in the data (EDA) |
| 6 | Prepare features for machine learning |
| 7 | Train 4 different ML models |
| 8 | Evaluate and compare all models |
| 9 | Use the best model to predict on a new startup |

---

## 📦 Step 1: Install & Import Dependencies

### What is this doing?

Before writing any code, we install and import all the **libraries** (pre-built tools) used throughout the notebook.

**Libraries being installed:**
- `kagglehub` — downloads datasets directly from Kaggle with one line
- `pandas` — loads and manipulates tabular data (like Excel, but in Python)
- `numpy` — fast math operations on arrays and matrices
- `matplotlib` / `seaborn` / `plotly` — data visualization and charting
- `scikit-learn` — the main ML toolkit (models, preprocessing, evaluation)
- `xgboost` — a powerful gradient boosting algorithm, often wins ML competitions
- `imbalanced-learn` — tools to handle datasets where one class is much rarer than the other

> 💡 The `!` before `pip install` tells the notebook to run a **terminal command** instead of Python code.

In [ ]:
# Install required packages
!pip install kagglehub pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn plotly -q

### Importing the libraries

Key imports explained:
- `train_test_split` — splits data into training set (model learns) and test set (we evaluate)
- `LabelEncoder` — converts text categories like `"USA"` into numbers like `42` (ML needs numbers)
- `StandardScaler` — rescales features so they're all on the same scale (important for Logistic Regression)
- `SimpleImputer` — fills in missing/NaN values automatically
- `SMOTE` — creates synthetic examples of the minority class to balance an imbalanced dataset
- `roc_auc_score` — measures how well the model separates the two classes (0.5 = random, 1.0 = perfect)
- `sns.set_theme(...)` — sets a consistent visual style for all charts

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# Style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 14, 'axes.labelsize': 12})

print('✅ All libraries imported successfully!')

---
## 🔑 Step 2: Download Dataset via kagglehub

### What is this doing?

We download the **Crunchbase VC Investments** dataset — a real-world dataset with thousands of startup records including their funding history, industry, country, and final outcome (acquired, IPO, closed, or still operating).

`kagglehub` is Kaggle's official Python library that downloads datasets with a single line of code — no manual file management needed.

**On first run**, it will prompt you to enter your Kaggle credentials:
- **Username** — your Kaggle account name
- **API Key** — get it from https://www.kaggle.com/settings → API → **Create New Token**

After that, credentials are cached and future runs won't ask again. The `path` variable holds the **local folder** where the dataset was saved.

In [ ]:
import kagglehub

# Downloads dataset and returns local path (prompts login on first run)
path = kagglehub.dataset_download('arindam235/startup-investments-crunchbase')
print('✅ Dataset downloaded to:', path)

### Listing downloaded files

`glob` is a Python tool for searching files by pattern. We use `**/*` to recursively list **all files** inside the downloaded folder, so we know exactly what CSV files are available to load.

In [ ]:
# List downloaded files
import glob
files = glob.glob(path + '/**/*', recursive=True)
for f in files:
    print(f)

---
## 📂 Step 3: Load & Inspect Data

### What is this doing?

We automatically find the CSV file inside the downloaded folder using `glob`, then load it into a **pandas DataFrame** — think of this as a Python spreadsheet where each row is a startup and each column is a property (funding amount, country, market, etc.).

**Why `encoding='latin-1'`?** Some characters in company names (accented letters etc.) aren't standard UTF-8, so this encoding handles them correctly.

**Why `low_memory=False`?** Tells pandas to read the full file before guessing column types, preventing mixed-type warnings on large files.

`.head()` shows the **first 5 rows** so we can see what the data looks like immediately.

In [ ]:
# Find CSV automatically from the kagglehub download path
import os, glob
csv_files = glob.glob(os.path.join(path, '**/*.csv'), recursive=True)
print('Found CSV files:', csv_files)

df = pd.read_csv(csv_files[0], encoding='latin-1', low_memory=False)

print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

### `.info()` — Column types and missing value counts

`df.info()` gives a **technical overview** of the dataset:
- Column names and data types (`object` = text, `float64` = decimal, `int64` = integer)
- How many **non-null** (non-missing) values each column has
- Total memory usage

This helps us spot which columns have lots of missing data and which are numeric vs text.

In [ ]:
df.info()

### `.describe()` — Statistical summary

`df.describe(include='all')` gives **summary statistics** for every column:
- For **numeric columns**: mean, min, max, standard deviation, quartiles
- For **text columns**: count, number of unique values, most frequent value

This quickly reveals outliers, skewed distributions, and near-constant columns.

In [ ]:
df.describe(include='all')

---
## 🧹 Step 4: Data Cleaning & Target Engineering

### 4a — Visualizing Missing Data

### What is this doing?

Real-world datasets are messy — many cells are empty (NaN = Not a Number). We need to understand *how much* data is missing per column before deciding what to do.

- `df.isnull().mean()` — calculates the **fraction of missing values** per column (0.0 = nothing missing, 1.0 = entirely missing)
- We filter to only columns that actually have some missing data, then plot as a bar chart

**Why does this matter?** A column that is 90% empty provides almost no useful signal — we should drop it rather than trying to guess most of its values.

In [ ]:
# ── Missing value overview ───────────────────────────────────────────────────
miss = df.isnull().mean().sort_values(ascending=False)
miss_pct = miss[miss > 0]

fig, ax = plt.subplots(figsize=(12, 5))
miss_pct.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title('Missing Data (%) per Column', fontweight='bold')
ax.set_ylabel('Missing Fraction')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4b — Dropping High-Missing Columns & Creating the Target Variable

### What is this doing?

**Part 1 — Drop columns with > 60% missing:**  
Any column missing more than 60% of its values is too sparse to be useful. We drop them automatically with a threshold filter.

**Part 2 — Create the target variable `success`:**  
This is the most important step — defining what "success" means for our model.

The dataset has a `status` column with 4 values:
- `operating` — still running (ambiguous outcome)
- `acquired` — bought by another company ✅ **SUCCESS**
- `ipo` — went public on a stock exchange ✅ **SUCCESS**
- `closed` — shut down ❌ **FAILURE**

We create a binary column `success` where **1 = acquired or IPO**, **0 = everything else**. This becomes the **label** our models will learn to predict.

In [ ]:
# ── Drop columns with > 60% missing ──────────────────────────────────────────
threshold = 0.60
df = df.loc[:, df.isnull().mean() < threshold]
print(f'Columns after dropping high-NaN cols: {df.shape[1]}')

# ── Define target: "success" = acquired OR ipo ──────────────────────────────
# The 'status' column contains: operating, acquired, closed, ipo
if 'status' in df.columns:
    df['success'] = df['status'].apply(
        lambda x: 1 if str(x).lower() in ['acquired', 'ipo'] else 0
    )
    print('Target distribution:')
    print(df['success'].value_counts())
else:
    print('Available columns:', df.columns.tolist())

---
## 📊 Step 5: Exploratory Data Analysis (EDA)

EDA is about **understanding your data visually** before building models. The goal is to spot patterns, outliers, and relationships that tell us what drives startup success.

---

### 5.1 — Target Class Distribution

### What is this doing?

Before any modeling, we must check **how balanced our dataset is**. If 95% of startups are labeled "not successful" and only 5% "successful", a lazy model could score 95% accuracy by always predicting failure — which is useless.

We plot two charts side by side:
- **Pie chart** — percentage split between successful and not successful
- **Bar chart** — raw counts with exact numbers labeled on top

This tells us whether we need to handle **class imbalance** later (we will — using SMOTE in Step 6).

In [ ]:
# ── 5.1 Target class distribution ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['success'].value_counts()
labels = ['Not Successful', 'Successful']
colors = ['#E07B54', '#4CAF81']

axes[0].pie(counts, labels=labels, autopct='%1.1f%%', colors=colors,
            startangle=140, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Startup Success Distribution', fontweight='bold')

axes[1].bar(labels, counts, color=colors, edgecolor='white', width=0.5)
axes[1].set_title('Count of Successful vs Not', fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[1].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 5.2 — Top 15 Markets by Success Rate

### What is this doing?

Not all industries are equal — some markets (biotech, enterprise software) have historically higher acquisition/IPO rates than others.

We:
1. Group startups by `market` (industry category)
2. Calculate the **success rate** (% that were acquired or IPO'd) per market
3. Filter to markets with at least 20 startups (to avoid misleading stats from tiny samples)
4. Plot the top 15 as a horizontal bar chart

**Why this matters:** Market choice is a strong signal for success — this helps investors and founders understand which sectors produce the most successful exits.

In [ ]:
# ── 5.2 Top 15 startup categories by success rate ────────────────────────────
cat_col = 'market' if 'market' in df.columns else 'category_list'

if cat_col in df.columns:
    cat_success = (
        df.groupby(cat_col)['success']
        .agg(['mean', 'count'])
        .query('count >= 20')
        .sort_values('mean', ascending=False)
        .head(15)
        .reset_index()
    )
    cat_success.columns = [cat_col, 'success_rate', 'count']

    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.barh(cat_success[cat_col][::-1],
                   cat_success['success_rate'][::-1] * 100,
                   color=sns.color_palette('viridis', 15))
    ax.set_xlabel('Success Rate (%)')
    ax.set_title(f'Top 15 Markets by Startup Success Rate', fontweight='bold')
    for bar in bars:
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

### 5.3 — Funding Rounds Analysis

### What is this doing?

A **funding round** is when a startup raises money from investors (Seed → Series A → Series B → ...). More rounds generally means investors kept believing in the company.

Two charts:
- **Left — Distribution**: How many funding rounds do most startups go through? Shows the typical fundraising journey.
- **Right — Success Rate vs Rounds**: Does more funding rounds actually increase chances of success? A line chart shows the success rate at each count of rounds.

**Expected insight:** More rounds = higher success rates, because each round is a new validation from investors.

In [ ]:
# ── 5.3 Funding rounds analysis ──────────────────────────────────────────────
if 'funding_rounds' in df.columns:
    df['funding_rounds'] = pd.to_numeric(df['funding_rounds'], errors='coerce')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(df['funding_rounds'].dropna(), bins=20, color='steelblue',
                 edgecolor='white', alpha=0.8)
    axes[0].set_title('Distribution of Funding Rounds', fontweight='bold')
    axes[0].set_xlabel('Number of Funding Rounds')
    axes[0].set_ylabel('Count')

    round_success = df.groupby('funding_rounds')['success'].mean() * 100
    round_success = round_success[round_success.index <= 15]
    axes[1].plot(round_success.index, round_success.values,
                 marker='o', color='#4CAF81', linewidth=2.5, markersize=8)
    axes[1].fill_between(round_success.index, round_success.values,
                         alpha=0.15, color='#4CAF81')
    axes[1].set_title('Success Rate by # of Funding Rounds', fontweight='bold')
    axes[1].set_xlabel('Funding Rounds')
    axes[1].set_ylabel('Success Rate (%)')

    plt.tight_layout()
    plt.show()

### 5.4 — Total Funding Amount vs Success

### What is this doing?

Does **raising more money** make a startup more likely to succeed?

The raw funding amounts span a huge range (\$1K to \$5B+), so we apply a **log transformation** (`log1p`) to compress the scale — otherwise a few billion-dollar outliers would dominate the chart and hide patterns in smaller startups.

- **Box plot (left)**: Median and spread of log-funding for successful vs not-successful startups. Higher box = more funding among winners.
- **Histogram (right)**: Overlapping distributions showing where successful and failed startups cluster in terms of funding raised.

**Expected insight:** Successful startups tend to have raised more total funding — but there's significant overlap, so it's not the only factor.

In [ ]:
# ── 5.4 Total funding amount vs success ──────────────────────────────────────
fund_col = 'funding_total_usd'
if fund_col in df.columns:
    df[fund_col] = pd.to_numeric(
        df[fund_col].astype(str).str.replace(',', ''), errors='coerce'
    )
    df_fund = df[df[fund_col].between(1, 5e9)].copy()
    df_fund['log_funding'] = np.log1p(df_fund[fund_col])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    df_fund.boxplot(column='log_funding', by='success', ax=axes[0],
                    patch_artist=True,
                    boxprops=dict(facecolor='lightblue'),
                    medianprops=dict(color='red', linewidth=2))
    axes[0].set_title('Log(Funding) by Success Class', fontweight='bold')
    axes[0].set_xlabel('Success (0=No, 1=Yes)')
    axes[0].set_ylabel('log(1 + Total Funding USD)')
    plt.sca(axes[0])
    plt.title('')

    for label, color in zip([0, 1], ['#E07B54', '#4CAF81']):
        subset = df_fund[df_fund['success'] == label]['log_funding']
        axes[1].hist(subset, bins=40, alpha=0.5, color=color, density=True,
                     label=['Not Successful', 'Successful'][label])
    axes[1].set_title('Funding Distribution by Outcome', fontweight='bold')
    axes[1].set_xlabel('log(1 + Total Funding USD)')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

### 5.5 — Top Countries by Startup Count & Success Rate

### What is this doing?

Geography matters — the startup ecosystem varies enormously by country. The US dominates in volume, but smaller countries can punch above their weight in success rates.

We use a **dual-axis chart** (two y-axes, one chart):
- **Blue bars (left axis)**: Total number of startups — shows which countries have the most startup activity
- **Green bars (right axis)**: Success rate (%) — shows which countries produce the most successful exits

We only include countries with at least 50 startups to ensure statistically meaningful comparisons.

In [ ]:
# ── 5.5 Top countries by startup count and success rate ──────────────────────
if 'country_code' in df.columns:
    country_stats = (
        df.groupby('country_code')['success']
        .agg(['mean', 'count'])
        .query('count >= 50')
        .sort_values('count', ascending=False)
        .head(20)
        .reset_index()
    )
    country_stats.columns = ['country', 'success_rate', 'total']

    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(country_stats))
    w = 0.4

    bars1 = ax.bar(x - w/2, country_stats['total'], w, label='Total Startups',
                   color='#5b8cdb', edgecolor='white')
    ax2 = ax.twinx()
    bars2 = ax2.bar(x + w/2, country_stats['success_rate'] * 100, w,
                    label='Success Rate %', color='#4CAF81', edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels(country_stats['country'], rotation=45, ha='right')
    ax.set_ylabel('Total Startups', color='#5b8cdb')
    ax2.set_ylabel('Success Rate (%)', color='#4CAF81')
    ax.set_title('Top 20 Countries: Startup Volume & Success Rate', fontweight='bold')

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    plt.tight_layout()
    plt.show()

### 5.6 — Correlation Heatmap

### What is this doing?

A **correlation heatmap** shows how strongly each pair of numeric features is related.

- Values range from **-1** (perfectly opposite) to **+1** (perfectly aligned), **0** = no relationship
- **Red/warm colors** = positive correlation (both go up together)
- **Green/cool colors** = negative correlation (one goes up, other goes down)
- We mask the upper triangle with `np.triu` since the matrix is symmetric — avoids redundancy

**What to look for:**
- Which features correlate most with `success`? Those are likely strong predictors.
- Two features highly correlated with *each other*? That's **multicollinearity** — can confuse some models.

In [ ]:
# ── 5.6 Correlation heatmap ───────────────────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'success' in num_cols:
    corr = df[num_cols].corr()

    mask = np.triu(np.ones_like(corr, dtype=bool))
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
                center=0, linewidths=0.5, ax=ax)
    ax.set_title('Feature Correlation Heatmap', fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 🔧 Step 6: Feature Engineering & Preprocessing

### What is this doing?

ML models can only work with **numbers** — not text, not missing values. This step transforms raw data into a clean numeric matrix the models can learn from.

### 6a — Select Features

We define which columns to use as **input features** (what the model sees when making predictions).

**Key features and what they mean:**
- `funding_rounds` — how many times the startup raised money (more = investors kept believing in it)
- `funding_total_usd` — total dollars raised (scale of investment)
- `country_code` — where the startup is based (ecosystem quality varies)
- `market` — what industry it's in (some industries have better exit rates)
- `has_VC`, `has_angel`, `has_roundA/B/C/D` — what types of investors backed it (tier of investors)
- `avg_participants` — average number of investors per round (more = more validation)
- `is_top500` — whether it appeared in a top startup ranking

In [ ]:
# ── Select features ───────────────────────────────────────────────────────────
feature_candidates = [
    'funding_rounds', 'funding_total_usd', 'country_code', 'market',
    'has_VC', 'has_angel', 'has_roundA', 'has_roundB', 'has_roundC',
    'has_roundD', 'avg_participants', 'is_top500'
]

# Keep only columns that exist
features = [c for c in feature_candidates if c in df.columns]
print('Selected features:', features)

# Also add numeric cols not in the list
extra_num = [c for c in num_cols if c not in features + ['success']]
features = features + extra_num
features = list(dict.fromkeys(features))  # deduplicate, preserve order

Xy = df[features + ['success']].copy()
print(f'Working dataset: {Xy.shape}')

### 6b — Encode, Impute, Split, SMOTE & Scale

### What is this doing?

The full preprocessing pipeline, done in this exact order:

**1. Label Encoding** — converts text like `country_code='USA'` → `42`. Models need numbers, not strings.

**2. Train/Test Split (80/20)** — splits data so:
- The model **learns** from 80% (training set)
- We **evaluate** on the remaining 20% (test set — model never sees this during training)
- `stratify=y` ensures both splits maintain the same success/failure ratio

**3. Imputation** — fills remaining missing values with the column **median**. We fit only on training data (fitting on test data would be "data leakage" — cheating).

**4. SMOTE** (Synthetic Minority Over-sampling Technique) — if only 20% of startups are "successful", the model is biased toward predicting failure. SMOTE creates **synthetic new examples** of successful startups by interpolating between real ones, balancing the training set.

**5. StandardScaler** — rescales each feature to mean=0, std=1. Critical for Logistic Regression but doesn't affect tree-based models. Always fit on training data only.

In [ ]:
# ── Encode categoricals, impute, scale ───────────────────────────────────────
le = LabelEncoder()
cat_feats = Xy.select_dtypes(include='object').columns.tolist()
for col in cat_feats:
    Xy[col] = le.fit_transform(Xy[col].astype(str))

X = Xy.drop(columns=['success'])
y = Xy['success']

# Impute missing
imp = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Class balance (train): {y_train.value_counts().to_dict()}')

In [ ]:
# ── Handle class imbalance with SMOTE ────────────────────────────────────────
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print(f'After SMOTE: {y_res.value_counts().to_dict()}')

# Scale
scaler = StandardScaler()
X_res_sc = scaler.fit_transform(X_res)
X_test_sc = scaler.transform(X_test)

---
## 🤖 Step 7: Train Multiple Models

### What is this doing?

We train **4 different ML models** and compare them. Using multiple models is best practice — no single algorithm is always best for every dataset.

| Model | How it works | Strengths |
|-------|-------------|----------|
| **Logistic Regression** | Fits a mathematical boundary between classes | Fast, interpretable, great baseline |
| **Random Forest** | Builds 150 decision trees, takes majority vote | Handles non-linear patterns, robust to outliers |
| **Gradient Boosting** | Builds trees sequentially, each correcting the previous one's mistakes | High accuracy on tabular data |
| **XGBoost** | Optimized, regularized gradient boosting with extra tricks | State-of-the-art for structured data, very fast |

**Important detail:**
- Logistic Regression uses **scaled data** (`X_res_sc`) — it's sensitive to feature magnitude
- Tree-based models use **unscaled data** (`X_res`) — they split on thresholds, scale doesn't affect them

For each model we compute **Accuracy** (% correct predictions) and **ROC-AUC** (how well it ranks successes above failures).

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=150, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=150, random_state=42,
                                          eval_metric='logloss', verbosity=0)
}

results = {}

for name, model in models.items():
    # Use scaled data for LR, raw for tree-based
    if 'Logistic' in name:
        model.fit(X_res_sc, y_res)
        preds = model.predict(X_test_sc)
        proba = model.predict_proba(X_test_sc)[:, 1]
    else:
        model.fit(X_res, y_res)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    results[name] = {'model': model, 'preds': preds, 'proba': proba,
                     'accuracy': acc, 'auc': auc}
    print(f'{name:25s} -> Accuracy: {acc:.4f}  |  ROC-AUC: {auc:.4f}')

---
## 📈 Step 8: Model Evaluation & Visualization

Training is only half the job — we need to deeply evaluate *how well* each model performs and *where* it makes mistakes.

---

### 8.1 — ROC Curves

### What is this doing?

The **ROC (Receiver Operating Characteristic) curve** plots:
- **X-axis**: False Positive Rate — how often we wrongly predict a startup as "successful"
- **Y-axis**: True Positive Rate — how often we correctly catch actual successes

A **perfect model** hugs the top-left corner. The **dashed diagonal** = random guessing (AUC = 0.5).

The **AUC** (Area Under Curve) summarizes performance in one number — the closer to 1.0, the better. Plotting all 4 models together makes comparison easy at a glance.

In [ ]:
# ── 8.1 ROC Curves ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
palette = ['#3B82F6', '#EF4444', '#10B981', '#F59E0B']

for (name, res), color in zip(results.items(), palette):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})",
            linewidth=2.5, color=color)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

### 8.2 — Model Comparison Bar Chart

### What is this doing?

A simple side-by-side bar chart comparing **Accuracy** and **ROC-AUC** for all 4 models simultaneously.

The **red dashed line at 0.8** is a common industry threshold for "good enough" performance. Models above this line are considered strong for real-world classification tasks.

This is the fastest way to identify which model to carry forward for final predictions.

In [ ]:
# ── 8.2 Model Comparison Bar Chart ───────────────────────────────────────────
metrics_df = pd.DataFrame([
    {'Model': n, 'Accuracy': r['accuracy'], 'ROC-AUC': r['auc']}
    for n, r in results.items()
]).set_index('Model')

fig, ax = plt.subplots(figsize=(10, 5))
metrics_df.plot(kind='bar', ax=ax, width=0.6, edgecolor='white',
                color=['#3B82F6', '#10B981'])
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=14)
ax.set_ylabel('Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.set_ylim(0, 1.1)
ax.axhline(0.8, linestyle='--', color='red', alpha=0.4, label='0.8 Threshold')
ax.legend()
plt.tight_layout()
plt.show()

### 8.3 — Confusion Matrices

### What is this doing?

A **confusion matrix** breaks down every prediction into 4 categories:

|  | Predicted: Not Successful | Predicted: Successful |
|--|--------------------------|----------------------|
| **Actually: Not Successful** | True Negative (TN) ✅ | False Positive (FP) ❌ |
| **Actually: Successful** | False Negative (FN) ❌ | True Positive (TP) ✅ |

- **True Positive (TP)**: Correctly predicted success — the model found a winner!
- **True Negative (TN)**: Correctly predicted failure — saved from a bad investment!
- **False Positive (FP)**: Predicted success but actually failed — wasted investment
- **False Negative (FN)**: Predicted failure but actually succeeded — missed opportunity!

All 4 models are shown in a 2×2 grid so you can visually compare where each model makes different kinds of errors.

In [ ]:
# ── 8.3 Confusion Matrices ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['preds'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Not Successful', 'Successful'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(name, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 8.4 — Feature Importance

### What is this doing?

Tree-based models can tell us **which features were most useful** for predictions. This is called **feature importance**.

We automatically pick the **best-performing model** by AUC and plot its top 15 most important features.

**How importance is calculated:** Every time a decision tree splits on a feature, it reduces prediction error. Features used more often and reducing error more get higher importance scores.

**Why this matters:**
- Tells us *what actually drives startup success* in the data
- Helps remove unimportant features in future model iterations
- Makes the model explainable to investors, founders, or business stakeholders

In [ ]:
# ── 8.4 Feature Importance (Best Tree Model) ─────────────────────────────────
best_name = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_name]['model']
print(f'Best model: {best_name} (AUC={results[best_name]["auc"]:.4f})')

if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=X_test.columns)
    fi = fi.sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(12, 6))
    fi.sort_values().plot(kind='barh', ax=ax,
                          color=sns.color_palette('crest', len(fi)))
    ax.set_title(f'Top 15 Feature Importances - {best_name}',
                 fontweight='bold', fontsize=14)
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

### 8.5 — Classification Report

### What is this doing?

The **classification report** gives a detailed per-class performance breakdown:

- **Precision**: Of all startups we *predicted* as successful, what fraction actually were? (reduces wasted investments)
- **Recall**: Of all startups that *actually* succeeded, what fraction did we correctly find? (reduces missed opportunities)
- **F1-Score**: The harmonic mean of precision and recall — a single balanced metric when both matter
- **Support**: How many real examples of each class exist in the test set

**The precision vs recall tradeoff:**
- Higher precision = fewer false positives (don't waste money on bad bets)
- Higher recall = fewer false negatives (don't miss great companies)

Depending on the use case, you may want to optimize for one over the other.

In [ ]:
# ── 8.5 Classification Report (Best Model) ───────────────────────────────────
print(f'\n📊 Classification Report - {best_name}\n')
print(classification_report(y_test, results[best_name]['preds'],
                             target_names=['Not Successful', 'Successful']))

---
## 🔮 Step 9: Predict on a New Startup

### What is this doing?

Now we use our trained best model to predict the success probability of a **brand new, hypothetical startup** it has never seen before.

**How it works step by step:**
1. Create a new row with the same columns as our training data, starting all at zero as a neutral baseline
2. Override specific features with realistic values (e.g. 3 funding rounds, \$5M raised)
3. Run the same preprocessing pipeline (impute → scale if Logistic Regression)
4. Call `predict_proba()` which returns a probability between 0 and 1
5. If probability >= 0.5 → predict **success**; otherwise → **not successful**

> 💡 **Try it yourself!** Change the values in `new_startup` to simulate different startups. Try increasing `funding_rounds` to 6 or `funding_total_usd` to 50M and watch the probability change!

In [ ]:
# Predict on a hypothetical new startup
# Fill in values matching your feature columns

new_startup = pd.DataFrame([
    {col: 0 for col in X_test.columns}  # Baseline: all zeros
])

# Override specific known features (adjust to your actual columns)
if 'funding_rounds' in new_startup.columns:
    new_startup['funding_rounds'] = 3
if 'funding_total_usd' in new_startup.columns:
    new_startup['funding_total_usd'] = 5_000_000

# Impute then scale if needed
new_imp = pd.DataFrame(imp.transform(new_startup), columns=X_test.columns)

if 'Logistic' in best_name:
    new_sc = scaler.transform(new_imp)
    prob = best_model.predict_proba(new_sc)[0][1]
else:
    prob = best_model.predict_proba(new_imp)[0][1]

print(f'\n🚀 Predicted Success Probability: {prob:.2%}')
print(f'   Verdict: {"✅ Likely Successful" if prob >= 0.5 else "❌ Likely Not Successful"}')

---
## ✅ Summary

| Step | What we did | Key concept |
|------|-------------|-------------|
| 1 | Installed & imported libraries | Dependencies |
| 2 | Downloaded Crunchbase dataset via `kagglehub` | Data acquisition |
| 3 | Loaded CSV, inspected shape, types, statistics | Data understanding |
| 4 | Dropped high-NaN columns, created binary `success` target | Data cleaning & labeling |
| 5 | Visualized class balance, markets, funding, countries, correlations | Exploratory Data Analysis |
| 6 | Label encoded categories, imputed NaNs, 80/20 split, SMOTE, scaled | Feature preprocessing |
| 7 | Trained Logistic Regression, Random Forest, Gradient Boosting, XGBoost | Model training |
| 8 | Evaluated with ROC-AUC, confusion matrices, feature importance, F1 scores | Model evaluation |
| 9 | Predicted success probability for a hypothetical new startup | Inference |

---

### 💡 Ideas to Improve Further

- **Hyperparameter tuning** — Use `GridSearchCV` or `RandomizedSearchCV` to find optimal model settings
- **Cross-validation** — Use `StratifiedKFold` for more robust, less lucky performance estimates
- **NLP features** — Extract signals from startup descriptions using TF-IDF or sentence embeddings
- **Time features** — Add founding year, years since last funding round as temporal signals
- **Model stacking** — Combine predictions from all 4 models using a meta-learner for better accuracy